# SBND RWM True Bucket: Extraction (Steps 2-6)

- True bucket b per segment from the Step 1 trace fits, the stored traces and the calibration CSVs (time chunks, segments)
- Writes `OUT_DIR/true_bucket_density.npz` (b per segment) + the caches `errors.ipynb` reads

## 1. Setup

- Set the paths in the Setup cell: `STORE`, `CALIB_CSV`, `OUT_DIR` (same as `trace_fits.ipynb`)

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pandas as pd
import pickle

import sys
sys.path.insert(0, '../lib')
import paths
from segments import select_chunks, segment_inputs
from ad_response import SEGMENTS, SEG_TITLE, AD_PERIODS, AD_TERMINATION, AD_B_S2, N_GROUPS, ad_term_products, load_ad_term
from true_bucket import (FF_PERIODS, train_end_tails, reflection_models, ad_waveforms_bootstrap, ft_context, ft_fits, save_npz, load_npz,
                         save_tails, load_tails)

# ===== Set these paths =====
STORE = {1: '/path/to/run1_rwm_traces', 2: '/path/to/run2_rwm_traces'}   # stored raw RWM traces: folders of run_<DAQ run>.npz
CALIB_CSV = {1: '/path/to/run1_calibration.csv', 2: '/path/to/run2_calibration.csv'}   # calibration CSVs from SBND_RWM_CALIBRATION
OUT_DIR = '../output'   # output folder, the same in every notebook
paths.set_paths(store=STORE, calib=CALIB_CSV, out=OUT_DIR)

F_TERM = paths.out_file('ad_term.npz')                # Step 2: end-of-spill probes per AD config
F_TAILS = paths.out_file('train_end.npz')             # Step 2: mean trace after the last bucket per AD config
F_WB = paths.out_file('corrected_overlays.npz')       # Step 2: per-bucket histograms of the corrected traces per segment
F_FITS = paths.out_file('true_bucket_fits.pkl')       # Steps 4-5: H, taps and group fits
F_DENS = paths.out_file('true_bucket_density.npz')    # Step 6: true bucket b per segment
F_CSV = paths.out_file('true_bucket.csv')             # Step 6: values + errors per segment
F_COV = paths.out_file('true_bucket_cov.npz')         # Step 6: cross-segment covariance of the errors

## 2. Time Chunks and Segments

- Traces per segment: good, unclipped, inside a time chunk of the segment with ≥ 5 such traces; one AD config per segment

In [ ]:
CHUNKS = {**select_chunks(1), **select_chunks(2)}
DATA = segment_inputs(CHUNKS)
pd.DataFrame({SEG_TITLE[k]: {'AD config': DATA[k]['blk_period'][0], 'scope input': AD_TERMINATION[DATA[k]['blk_period'][0]],
                             'time chunks': len(CHUNKS[k]['sel']), 'traces': len(CHUNKS[k]['ad']), 'period P [ns]': f"{DATA[k]['P']:.5f}"}
              for k in SEGMENTS}).T

## 3. Step 2: Remove Cable Reflections

- Round trip per AD config from the end-of-spill probes of every stored trace; short reflection from the mean trace after the last bucket
- Every trace of the segments corrected, bucket positions refit on the corrected trace, per-bucket histograms of the corrected trace

In [ ]:
ad_term_products(F_TERM)
TERM = load_ad_term(F_TERM)
TERM_PERIODS = [pn for pn in AD_PERIODS if pn in TERM and len(TERM[pn]['psd']) >= 100]     # AD config 1.2: clipped, a few traces only
assert TERM_PERIODS == list(FF_PERIODS), TERM_PERIODS
save_tails(F_TAILS, train_end_tails())
TAILS = load_tails(F_TAILS)
RMOD = reflection_models(TERM, TAILS, TERM_PERIODS)
WB, _ = ad_waveforms_bootstrap(RMOD)
save_npz(F_WB, WB, compress=True)
WB = load_npz(F_WB)
pd.DataFrame({pn: {'scope input': AD_TERMINATION[pn], 'round trip Σh': f"{RMOD[pn]['h'].sum():+.4f}",
                   'short reflection T [ns] / ρ': f"{RMOD[pn]['short']['T']:.2f} / {RMOD[pn]['short']['rho']:+.3f}"} for pn in TERM_PERIODS})

## 4. Steps 3-5: Groups, AD Response, Forward Fit

- Buckets aligned into 4 groups per segment; H (AD config 2.1) and the correction taps fitted; per group A · rebin(b ⊗ R_AD) in ±2.5 σ_AD

In [ ]:
CTX = ft_context(DATA, WB)
FIT = {k: ft_fits(CTX, k) for k in SEGMENTS}
with open(F_FITS, 'wb') as f_:
    pickle.dump(dict(HF=CTX['HF'], RTF=CTX['RTF'], FIT=FIT), f_)
pd.DataFrame({SEG_TITLE[k]: {f'group {m}: σL / σR / βL / βR, χ²/ndof': (lambda g: f"{g['p']['sl']:.3f} / {g['p']['sr']:.3f} / {g['p']['bl']:.2f} / "
                                                                          f"{g['p']['br']:.2f}, {g['chi2'] / g['ndof']:.2f}")(FIT[k]['fg'][m])
                             for m in range(N_GROUPS)} for k in SEGMENTS}).T

## 5. Step 6: True Bucket per Segment

- b = weighted mixture of the 4 group b ⊗ G(s²) (s² > 0 for AD config 2.1 only); density on a fine grid, mode at 0

In [ ]:
save_npz(F_DENS, {k: dict(tt=FIT[k]['mix']['tt'], ff=FIT[k]['mix']['ff'], P=CTX['DSF'][k]['grid'].P) for k in SEGMENTS})
pd.DataFrame({SEG_TITLE[k]: {'s² [ns²]': AD_B_S2[DATA[k]['blk_period'][0]], **{n: f"{FIT[k]['mix']['agg2'][n]:.3f}" for n in ('sl', 'sr', 'bl', 'br')},
                             'RMS [ns]': f"{FIT[k]['mix']['rms']:.3f}"} for k in SEGMENTS}).T